In [0]:
from pyspark.sql.functions import col, when, current_timestamp

bronze_df = spark.readStream.table('fraud_mlops.bronze.fraud_transactions_incremental')

silver_base_df = bronze_df.select(
    col("step"),
    col("type").alias("transaction_type"),
    col("amount"),
    col("nameOrig").alias("origin_account"),
    col("oldbalanceOrg").alias("old_balance_origin"),
    col("newbalanceOrig").alias("new_balance_origin"),
    col("nameDest").alias("destination_account"),
    col("oldbalanceDest").alias("old_balance_destination"),
    col("newbalanceDest").alias("new_balance_destination"),
    col("isFraud").alias("is_fraud"),
    col("isFlaggedFraud").alias("is_flagged_fraud"),
    col("source_file"),
    col("ingested_at")
)

transaction_columns = [

    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "old_balance_origin",
    "new_balance_origin",
    "destination_account",
    "old_balance_destination",
    "new_balance_destination",
    "is_fraud",
    "is_flagged_fraud"

]

silver_base_df = silver_base_df.dropDuplicates(transaction_columns)

allowed_types = [

    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER"

]

checked_df = (
    silver_base_df

    .withColumn(
        "dq_reason",

        when(
            col("step").isNull(),
            "missing_step"
        )

        .when(
            ~col("step").between(1, 744),
            "invalid_step"
        )

        .when(
            col("transaction_type").isNull(),
            "missing_transaction_type"
        )

        .when(
            ~col("transaction_type").isin(allowed_types),
            "invalid_transaction_type"
        )

        .when(
            col("amount").isNull(),
            "missing_amount"
        )

        .when(
            col("amount") < 0,
            "negative_amount"
        )

        .when(
            col("origin_account").isNull(),
            "missing_origin_account"
        )

        .when(
            col("destination_account").isNull(),
            "missing_destination_account"
        )

        .when(
            col("is_fraud").isNotNull()
            & ~col("is_fraud").isin(0, 1),
            "invalid_fraud_label"
        )

        .when(
            col("is_flagged_fraud").isNotNull()
            & ~col("is_flagged_fraud").isin(0, 1),
            "invalid_flagged_fraud"
        )
    )
    .withColumn(
        "label_status",
        when(
            col("is_fraud").isNull(),
            "PENDING"
        ).otherwise("AVAILABLE")
    )
    .withColumn(
        "dq_status",
        when(
            col("dq_reason").isNotNull(),
            "QUARANTINE"
        ).otherwise("VALID")
    )
)


clean_df = checked_df.filter(col('dq_status') == 'VALID')\
    .drop('dq_reason','dq_status')\
        .withColumn('silver_processed_at',current_timestamp())


quarrentine_df = checked_df.filter(col('dq_status')== 'QUARANTINE')\
    .withColumn('quarantined_at',current_timestamp())


clean_query = clean_df.writeStream.format('delta')\
    .outputMode('append')\
    .option('checkpointLocation',\
        'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/checkpoints/transaction_clean_silver')\
    .trigger(availableNow=True)\
    .toTable('fraud_mlops.silver.transactions_clean_incremental')


quarantine_query = quarrentine_df.writeStream.format('delta')\
    .outputMode('append')\
    .option('checkpointLocation',\
        'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/checkpoints/transaction_quarantine_silver')\
    .trigger(availableNow=True)\
    .toTable('fraud_mlops.silver.transactions_quarantine_incremental')


clean_query.awaitTermination()
quarantine_query.awaitTermination()

print("Bronze -> Silver incremental processing completed.")

In [0]:
%sql
SELECT COUNT(*) FROM fraud_mlops.silver.transactions_clean_incremental